# Summary - Accelearting end-to-end data flows

In [4]:
#%pip install cudf-cu12 dask-cudf-cu12 --extra-index-url=https://pypi.ngc.nvidia.com
#%pip install cupy --upgrade

In [ ]:
# set working directory to downloads folder
import os
import cudf
import cupy as cp
import numpy as np

from datetime import datetime
import random
import time

os.chdir('/Users/veronicalarsson/Downloads')

## Section 1: Accelerated Data Manipulation

### 2. Data Manipulation: cuDF vs Pandas

In data science, **pandas** is used for: 
* **Data loading and writing**: reads from and writes to various file formats like CSV, Excel, JSON, and SQL databases
* **Data cleaning and processing/preprocessing**: helps users with handling missing data, merging datasets, and reshaping data
* **Data analysis**: performs grouping, aggregating, and statistical operations

**Note**: Data preprocessing refers to the process of transforming raw data into a format that is more suitable for analysis and other downstream tasks. 

The key features of **cuDF** include: 
* **GPU Acceleration**: leverages NVIDIA GPUs for fast data processing and analysis
* **pandas-like API**: provides users a familiar interface and transition to GPU-based computing
* **Integration with other RAPIDS libraries**: works seamlessly with other GPU-accelerated tools in the RAPIDS ecosystem

In [ ]:
# CREATING A GPU DATAFRAME WITH cuDF
#df = cudf.read_csv('./data/uk_pop.csv') #read data
df = cudf.read_csv('./data/uk_pop.csv') #read data

# BASIC DATAFRAME OPERATIONS
display(df.loc[0, 'age']) #allows indexing like pandas with loc

# EXERCISE 1: STRING OPERATIONS
df['county'].str.title() #convert county column content to title case (title -> Title)

# BOOLEAN INDEXING
df['age']>=18 #boolean indexing to check if age is greater than or equal to 18

# AGGREGATIONS
df[['lat', 'long']].mean()

# APPLYING USER-DEFINED FUNCTIONS (UDFs) WITH .apply() AND .map()
def is_adult(row): 
    if row['age']>=18: 
        return 1
    else: 
        return 0

display(df.apply(is_adult, axis=1)) #apply UDF to each row to check if age is 18 or older
display(df.apply(lambda x: 1 if x['age']>=18 else 0, axis=1)) #same as above but using a lambda function

# FILTERING WITH BOOLEAN MASKS AND .LOC
boolean_mask=df['name'].str.startswith('E')
df.loc[boolean_mask]

# COMBINING CONDITIONS WITH & AND |
df[(df['age']>=18) | (df['name'].str.startswith('E'))] #filter rows where age is 18 or older OR name starts with 'E'
df[(df['age']>=18) & (df['name'].str.startswith('E'))] #filter rows where age is 18 or older AND name starts with 'E'

# EXERCISE 2: FINDING COUNTIES NORTH OF SUNDERLAND
sunderland_residents=df.loc[df['county'] == 'SUNDERLAND'] #filter rows where county is SUNDERLAND
northmost_sunderland_lat=sunderland_residents['lat'].max() #find maximum latitude among SUNDERLAND residents
df.loc[df['lat'] > northmost_sunderland_lat]['county'].unique() #find unique counties north of SUNDERLAND

# CREATING NEW COLUMNS
current_year=datetime.now().year # get current year

df['birth_year']=current_year-df['age'] # numerical operations

df['sex_normalize']=df['sex'].str.upper() # string operations
df['county_normalize']=df['county'].str.title().str.replace(' ', '_')
df['name']=df['name'].str.title()

**cuDF pandas**

cuDF introduced a **pandas accelerator mode** (`cudf.pandas`) that supports 100% of the pandas API. **This mode allows users to accelerate pandas code on the GPU without requiring any code changes.** **Not all operations can be performed on the GPU.** When using `cudf.pandas`, **operations that can be accelerated will run on the GPU, while unsupported operations will automatically fall back to pandas on the CPU**. For example, `.read_sql()`. this will first read sql with pandas and move the data to cuDF. 

There are two ways to activate cuDF pandas:
- Jupyter Magic Command
```
%load_ext cudf.pandas
import pandas
...
```
- Python Import
```
import cudf.pandas
cudf.pandas.install()

import pandas as pd
...
```

**Note**: There are no other changes required - this is useful to quickly accelerate existing workloads with minimum code change. More information about cuDF pandas can be found [here](https://docs.rapids.ai/api/cudf/stable/cudf_pandas/). 

cuDF pandas is a no code change accelerator for pandas for automatic acceleration of any supported pandas call. 

Automatic Acceleration
* Uncomment the `%load_ext` magic command to accelerate with cuDF pandas. Observe the acceleration. 
* Uncomment the `%%cudf.pandas.line_profile` magic command to use the line profiler. Observe the output from the line profiler. 

In [ ]:
# %load_ext cudf.pandas #uncomment to activate cuDF pandas in Jupyter
import pandas as pd
import time
from datetime import datetime

# %%cudf.pandas.line_profile
start=time.time()

df=pd.read_csv('./data/uk_pop.csv')
current_year=datetime.now().year

df['birth_year']=current_year-df['age']

df['sex_normalize']=df['sex'].str.upper()
df['county_normalize']=df['county'].str.title().str.replace(' ', '_')
df['name']=df['name'].str.title()

print(f'Duration: {round(time.time()-start, 2)} seconds')

display(df.head())

### 3. Memory Management

This notebook explores the dynamics between data and memory.

During the data acquisition process, data is transferred to memory in order to be operated on by the processor. Memory management is crucial for cuDF and GPU operations for several key reasons: 
* **Limited GPU memory**: GPUs typically have less memory than CPUs, therefore efficient memory management is essential to maximize the use of available GPU memory, especially for large datasets.
* **Data transfer overhead**: Transferring data between CPU and GPU memory is relatively slow compared to GPU computation speed. Minimizing these transfers through smart memory management is critical for performance.
* **Performance tuning**: Understanding and optimizing memory usage is key to achieving peak performance in GPU-accelerated data processing tasks.

When done correctly, keeping the data on the GPU can enable cuDF and the RAPIDS ecosystem to achieve significant performance improvements, handle larger datasets, and provide more efficient data processing capabilities. 

In [ ]:
# pandas memory utilization
df=pd.read_csv('./data/uk_pop.csv')
mem_usage_df=df.memory_usage(deep=True) # We can use `DataFrame.memory_usage()` to see the memory usage for each column (in bytes).
mem_usage_df

# 64-bit numbers uses 8 bytes of memory
mem_usage_df[mem_usage_df.index.str.contains('64')] = mem_usage_df[mem_usage_df.index.str.contains('64')] * 8 # convert to bytes

# nan==nan when value is not a number
# nan uses 32 bytes of memory!

# total memory usage in bytes -> units in the power of 2 (more common than previous memory usage: to units in power of 10)
suffixes = ['B', 'kB', 'MB', 'GB', 'TB', 'PB']
def make_decimal(nbytes):
    i=0
    while nbytes >= 1024 and i < len(suffixes)-1:
        nbytes/=1024.
        i+=1
    f=('%.2f' % nbytes).rstrip('0').rstrip('.')
    return '%s %s' % (f, suffixes[i])

make_decimal(mem_usage_df.sum())

**Efficient Data Loading by (1) using cuDF pandas & (2) specifying column data types to lower bit types**

*By default, pandas (and cuDF) uses 64-bit for numerical values.* Using 64-bit numbers provides the highest precision but many applications do not require 64-bit precision when aggregating over a very large number of data points. When possible, using 32-bit numbers reduces storage and memory requirements in half, and also typically greatly speeds up computations because only half as much data needs to be accessed in memory. 

It is often advantageous to specify the most appropriate data types for each columns, based on range, precision requirement, and how they are used. 

In [ ]:
# by default pandas uses 64-bit for numerical values, to reduce memory usage we can convert to 32-bit
df['age']=df['age'].astype('int8') #numerical column to 8-bit integer
df['lat']=df['lat'].astype('float32') #numerical column to 32-bit float
df.dtypes

# convert categorical columns to 'category' data type which means the column contains a fixed number of possible values. 
# When appropriate, using the `categorical` data type can reduce memory usage and lead to faster operations. 
# It can also be used to define and maintain a custom order of categories.
df['sex']=df['sex'].astype('category')
df['county']=df['county'].astype('category')

In [ ]:
%load_ext cudf.pandas # `cuda.pandas` enabled in order to read data efficiently, without requiring any code changes
import pandas as pd

# define data types for each column
dtype_dict={
    'age': 'int8', 
    'sex': 'category', 
    'county': 'category', 
    'lat': 'float64', 
    'long': 'float64', 
    'name': 'category'
}
        
efficient_df=pd.read_csv('./data/uk_pop.csv', dtype=dtype_dict)
duration=time.time()-start

mem_usage_df=efficient_df.memory_usage('deep')
display(mem_usage_df)

print(f'Loading {make_decimal(mem_usage_df.sum())} took {round(duration, 2)} seconds.')

### 4. Interoperability

Examples of how we can use cuDF and CuPy together to *take advantage of CuPy array functionality* (such as advanced linear algebra operations). 



In [6]:
# 1: Get the CuPy representation of the data frame

#a: From cuDF, the `DataFrame.values`

#b. Convert via the CUDA array interface using `DataFrame.to_cupy()`

#c. Pass the Series to the `cupy.asarray()` function

## Section 2 - GPU-Accelerated Graph Analytics